In [ ]:
import pandas as pd
import os
import pyarrow as pa
import pyarrow.parquet as pq
import re

def convert_csv_to_parquet(csv_file_path):
    """
    Convert a CSV file to Parquet format and save it in the same folder.
    
    Parameters:
    -----------
    csv_file_path : str
        Path to the CSV file to be converted
    
    Returns:
    --------
    str
        Path to the created Parquet file
    """
    # Extract directory and filename
    directory = os.path.dirname(csv_file_path)
    filename = os.path.basename(csv_file_path)
    
    # Create output filename (replace extension with .parquet)
    output_filename = os.path.splitext(filename)[0] + '.parquet'
    output_path = os.path.join(directory, output_filename)
    
    print(f"Reading CSV file: {csv_file_path}")
    
    # Read the CSV file
    # Based on the analysis, this file has a non-standard structure with a header line
    # followed by a line with column names and then data
    
    # First, check if the file exists
    if not os.path.exists(csv_file_path):
        raise FileNotFoundError(f"File not found: {csv_file_path}")
    
    # Read the first few lines to understand structure
    with open(csv_file_path, 'r', encoding='utf-8') as f:
        first_line = f.readline().strip()
        second_line = f.readline().strip()
    
    # Extract the actual column names from the second line
    # The second line contains the actual data column names
    column_names = second_line.split(',')
    
    # Clean column names (remove quotes and extra whitespace)
    column_names = [re.sub(r'^"|"$', '', col.strip()) for col in column_names]
    
    # Read the CSV file, skipping the first 2 lines (header and column names)
    # and using the extracted column names
    df = pd.read_csv(
        csv_file_path, 
        skiprows=2,  # Skip the first two rows
        names=column_names,  # Use the column names from the second line
        low_memory=False
    )
    
    print(f"CSV file loaded successfully. Shape: {df.shape}")
    
    # Convert to Parquet
    print(f"Converting to Parquet format...")
    
    # Convert to PyArrow Table and write to Parquet
    table = pa.Table.from_pandas(df)
    pq.write_table(
        table, 
        output_path, 
        compression='snappy'  # 'snappy' is a good default compression
    )
    
    print(f"Conversion complete. Parquet file saved to: {output_path}")
    return output_path

if __name__ == "__main__":
    import argparse
    
    parser = argparse.ArgumentParser(description='Convert CSV file to Parquet format.')
    parser.add_argument('csv_file', help='Path to the CSV file to convert')
    args = parser.parse_args()
    
    try:
        output_path = convert_csv_to_parquet(args.csv_file)
        print(f"Successfully converted to: {output_path}")
    except Exception as e:
        print(f"Error: {e}")